In [ ]:
# pacer_weighted_bart.py
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import BartForConditionalGeneration, BartTokenizerFast
from transformers.models.bart.modeling_bart import BartAttention

In [ ]:
# ----------------------------
# 1) Sentence weights alpha
# ----------------------------
def compute_alpha_from_relevance(
    w: torch.Tensor,          # (K,)
    phi: torch.Tensor,        # (m, K)
    tau_alpha: float = 2.0
) -> torch.Tensor:
    """
    alpha_i = softmax(tau_alpha * (phi_i^T w))
    """
    # relevance: (m,)
    rel = phi @ w
    alpha = torch.softmax(tau_alpha * rel, dim=0)
    return alpha


In [ ]:
# ----------------------------
# 2) Tokenize selected sentences + spans
# ----------------------------
@dataclass
class TokenizedSentences:
    input_ids: torch.Tensor              # (1, seq_len)
    attention_mask: torch.Tensor         # (1, seq_len)
    sentence_spans: List[Tuple[int,int]] # [(start,end), ...] token indices in the *encoder* sequence


def tokenize_sentences_with_spans_bart(
    sentences: List[str],
    tokenizer: BartTokenizerFast,
    max_length: int = 1024,
) -> TokenizedSentences:
    """
    Build a single encoder input by concatenating sentences with a separator token,
    and return token spans for each sentence.

    IMPORTANT: spans are computed in terms of token positions in the encoder input_ids.
    """
    assert tokenizer.bos_token_id is not None and tokenizer.eos_token_id is not None

    sep_id = tokenizer.eos_token_id  # BART commonly uses </s> as a separator

    # We will manually build tokens to control spans
    pieces: List[int] = [tokenizer.bos_token_id]
    spans: List[Tuple[int,int]] = []

    cur = 1  # current token index after BOS
    for s in sentences:
        ids = tokenizer.encode(s, add_special_tokens=False)
        start = cur
        pieces.extend(ids)
        cur += len(ids)
        end = cur  # end exclusive
        spans.append((start, end))

        # add separator after each sentence
        pieces.append(sep_id)
        cur += 1

    # final EOS
    pieces.append(tokenizer.eos_token_id)

    # truncate (keep spans safe)
    if len(pieces) > max_length:
        pieces = pieces[:max_length]
    seq_len = len(pieces)

    # clamp spans to seq_len
    clamped_spans = []
    for (a, b) in spans:
        a = max(0, min(a, seq_len))
        b = max(0, min(b, seq_len))
        if a < b:
            clamped_spans.append((a, b))
        else:
            clamped_spans.append((0, 0))  # empty span if truncated away

    input_ids = torch.tensor(pieces, dtype=torch.long).unsqueeze(0)  # (1, seq_len)
    attention_mask = torch.ones_like(input_ids)

    return TokenizedSentences(input_ids=input_ids, attention_mask=attention_mask, sentence_spans=clamped_spans)



In [ ]:
# ----------------------------
# 3) Sentence weights -> token bias
# ----------------------------
def build_token_bias_from_sentence_weights(
    seq_len: int,
    sentence_spans: List[Tuple[int,int]],
    alpha: torch.Tensor,           # (m,)
    lambda_att: float = 1.0,
    eps: float = 1e-8,
    device=None,
) -> torch.Tensor:
    """
    Returns bias_key: shape (seq_len,), where bias_key[pos] = lambda_att * log(alpha[sent(pos)] + eps)
    Tokens not in any span get 0.
    """
    if device is None:
        device = alpha.device

    bias_key = torch.zeros(seq_len, device=device, dtype=torch.float32)
    logw = lambda_att * torch.log(alpha.to(device=device, dtype=torch.float32) + eps)

    for sent_id, (a, b) in enumerate(sentence_spans):
        a = max(0, int(a))
        b = min(seq_len, int(b))
        if a < b:
            bias_key[a:b] = logw[sent_id]
    return bias_key

In [ ]:
# ----------------------------
# 4) Patch BART cross-attention to add bias
# ----------------------------
class BartAttentionWithKeyBias(BartAttention):
    """
    A BartAttention that reads a stored 1D bias over key positions:
        self._pacer_bias_key  shape: (src_len,)
    and adds it to attention logits before softmax:
        attn_weights[b,h,q,s] += bias_key[s]
    """

    def forward(
        self,
        hidden_states,
        key_value_states=None,
        past_key_value=None,
        attention_mask=None,
        layer_head_mask=None,
        output_attentions=False,
    ):
        bsz, tgt_len, _ = hidden_states.size()
        query_states = self.q_proj(hidden_states) * self.scaling

        is_cross_attention = key_value_states is not None
        if is_cross_attention:
            key_states = self.k_proj(key_value_states)
            value_states = self.v_proj(key_value_states)
        else:
            key_states = self.k_proj(hidden_states)
            value_states = self.v_proj(hidden_states)

        def shape(x, seq_len):
            return x.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2).contiguous()

        query_states = shape(query_states, tgt_len)
        src_len = key_states.size(1)
        key_states = shape(key_states, src_len)
        value_states = shape(value_states, src_len)

        attn_weights = torch.matmul(query_states, key_states.transpose(-2, -1))  # (bsz, heads, tgt_len, src_len)

        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask

        # ---------- NEW: add key-position bias if present ----------
        bias_key = getattr(self, "_pacer_bias_key", None)
        if bias_key is not None:
            # bias_key: (src_len,)
            if bias_key.dim() != 1 or bias_key.shape[0] != src_len:
                raise ValueError(f"bias_key must be (src_len,), got {tuple(bias_key.shape)} vs src_len={src_len}")
            # reshape to (1,1,1,src_len) then broadcast across (bsz, heads, tgt_len, src_len)
            attn_weights = attn_weights + bias_key.view(1, 1, 1, src_len)

        if layer_head_mask is not None:
            attn_weights = attn_weights * layer_head_mask

        attn_probs = F.softmax(attn_weights, dim=-1)
        attn_probs = F.dropout(attn_probs, p=self.dropout, training=self.training)

        attn_output = torch.matmul(attn_probs, value_states)
        attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, tgt_len, self.embed_dim)
        attn_output = self.out_proj(attn_output)

        if output_attentions:
            return attn_output, attn_probs, past_key_value
        return attn_output, None, past_key_value


def patch_bart_cross_attention(model: BartForConditionalGeneration) -> BartForConditionalGeneration:
    """
    Replace decoder cross-attention blocks with BartAttentionWithKeyBias.
    """
    for layer in model.model.decoder.layers:
        old = layer.encoder_attn
        new = BartAttentionWithKeyBias(
            embed_dim=old.embed_dim,
            num_heads=old.num_heads,
            dropout=old.dropout,
            is_decoder=True,
            bias=old.k_proj.bias is not None
        )
        new.load_state_dict(old.state_dict(), strict=True)
        layer.encoder_attn = new
    return model


def set_pacer_key_bias_on_model(model: BartForConditionalGeneration, bias_key: Optional[torch.Tensor]):
    """
    Attach bias_key to every decoder cross-attention module.
    bias_key should be (src_len,) on the same device as encoder inputs.
    """
    for layer in model.model.decoder.layers:
        layer.encoder_attn._pacer_bias_key = bias_key




In [ ]:
# ----------------------------
# 5) End-to-end usage example
# ----------------------------
@torch.no_grad()
def generate_weighted_summary_bart(
    model: BartForConditionalGeneration,
    tokenizer: BartTokenizerFast,
    selected_sentences: List[str],  # your S_t (texts)
    w_user: torch.Tensor,           # (K,)
    phi_selected: torch.Tensor,     # (m,K) aspect vectors aligned with selected_sentences
    tau_alpha: float = 2.0,
    lambda_att: float = 1.0,
    max_new_tokens: int = 80,
    device: str = "cuda",
) -> str:
    model = model.to(device)
    model.eval()

    # 1) tokenize + spans
    tok = tokenize_sentences_with_spans_bart(selected_sentences, tokenizer)
    input_ids = tok.input_ids.to(device)
    attention_mask = tok.attention_mask.to(device)

    # 2) alpha weights
    w_user = w_user.to(device)
    phi_selected = phi_selected.to(device)
    alpha = compute_alpha_from_relevance(w_user, phi_selected, tau_alpha=tau_alpha)

    # 3) bias_key over encoder token positions
    seq_len = input_ids.shape[1]
    bias_key = build_token_bias_from_sentence_weights(
        seq_len=seq_len,
        sentence_spans=tok.sentence_spans,
        alpha=alpha,
        lambda_att=lambda_att,
        device=device,
    )

    # 4) attach bias into cross-attention modules
    set_pacer_key_bias_on_model(model, bias_key)

    # 5) generate
    out_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        num_beams=4,
    )

    # 6) remove bias (important so it doesn't leak to next call)
    set_pacer_key_bias_on_model(model, None)

    return tokenizer.decode(out_ids[0], skip_special_tokens=True)


In [ ]:
# if __name__ == "__main__":
    # Example
tokenizer = BartTokenizerFast.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
model = patch_bart_cross_attention(model)

# Your selected sentences S_t (texts)
S_t = [
    "Battery life is excellent and lasts two days.",
    "The camera quality is average in low light.",
    "Shipping was fast and packaging was good."
]

# Example: K aspects
K = 4
# user preference vector w (simplex)
w_user = torch.tensor([0.7, 0.1, 0.1, 0.1], dtype=torch.float32)  # prefers aspect 1 heavily

# Example phi for selected sentences (m,K)
# Suppose aspect 1=battery, 2=camera, 3=shipping, 4=build
phi_selected = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],  # sentence1 is battery
    [0.0, 1.0, 0.0, 0.0],  # sentence2 is camera
    [0.0, 0.0, 1.0, 0.0],  # sentence3 is shipping
], dtype=torch.float32)

summary = generate_weighted_summary_bart(
    model=model,
    tokenizer=tokenizer,
    selected_sentences=S_t,
    w_user=w_user,
    phi_selected=phi_selected,
    tau_alpha=3.0,
    lambda_att=2.0,
    max_new_tokens=80,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print(summary)